In [10]:
import os
import random
import math
import numpy as np
import pandas as pd
from collections import Counter

from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

H, W = 128, 800
N_PIXELS = H * W
N_IMAGES = 1200

DATA_DIR = "/kaggle/input/competitions/industrial-surface-defect-detection/public"
CSV_PATH = os.path.join(DATA_DIR, "train.csv")

df = pd.read_csv(CSV_PATH)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

Shape: (45572, 5)
Columns: ['image_id', 'defect_type', 'mask_rle', 'width', 'height']


,image_id,defect_type,mask_rle,width,height
0,4S22I7SA34,defect_a,7255 6 7377 12 7505 12 7633 12 7762 11 7890 11...,800,128
1,4S22I7SA34,defect_b,NaN,800,128
2,4S22I7SA34,defect_c,NaN,800,128
3,4S22I7SA34,defect_d,NaN,800,128
4,DZ899111OE,defect_a,NaN,800,128


In [11]:
print("CSV columns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nData types:")
print(df.dtypes)

CSV columns:
['image_id', 'defect_type', 'mask_rle', 'width', 'height']

First 5 rows:


,image_id,defect_type,mask_rle,width,height
0,4S22I7SA34,defect_a,7255 6 7377 12 7505 12 7633 12 7762 11 7890 11...,800,128
1,4S22I7SA34,defect_b,NaN,800,128
2,4S22I7SA34,defect_c,NaN,800,128
3,4S22I7SA34,defect_d,NaN,800,128
4,DZ899111OE,defect_a,NaN,800,128



Data types:
image_id       object
defect_type    object
mask_rle       object
width           int64
height          int64
dtype: object


In [12]:
image_ids = sorted(df["image_id"].dropna().unique().tolist())[:1200]

subset = df[df["image_id"].isin(image_ids)].copy()

subset["image_id"] = subset["image_id"].astype(str)

subset = subset.sort_values(
    ["image_id", "defect_type"]
).reset_index(drop=True)

assert len(image_ids) == 1200
assert len(subset) == 4800

print("Unique images:", subset["image_id"].nunique())
print("Image-class rows:", len(subset))
print("Classes:", sorted(subset["defect_type"].unique()))

display(subset.head())

Unique images: 1200
Image-class rows: 4800
Classes: ['defect_a', 'defect_b', 'defect_c', 'defect_d']


,image_id,defect_type,mask_rle,width,height
0,000HAM628W,defect_a,NaN,800,128
1,000HAM628W,defect_b,NaN,800,128
2,000HAM628W,defect_c,80866 7 80980 23 81094 40 81207 57 81325 69 81...,800,128
3,000HAM628W,defect_d,NaN,800,128
4,005VFS2MW4,defect_a,NaN,800,128


In [13]:
def decode_rle(rle, height=128, width=800):
    mask = np.zeros(height * width, dtype=np.uint8)

    if pd.isna(rle) or str(rle).strip() == "":
        return mask.reshape((height, width), order="F")

    values = np.asarray(str(rle).split(), dtype=np.int64)

    starts = values[0::2] - 1
    lengths = values[1::2]

    ends = np.minimum(starts + lengths, height * width)

    for start, end in zip(starts, ends):
        start = max(0, start)
        if end > start:
            mask[start:end] = 1

    return mask.reshape((height, width), order="F")


mask_lookup = {}

for image_id, group in subset.groupby("image_id", sort=False):
    masks = {}

    for _, row in group.iterrows():
        defect = str(row["defect_type"])
        masks[defect] = decode_rle(row["mask_rle"])

    mask_lookup[image_id] = masks

classes = sorted(subset["defect_type"].unique().tolist())

print("Decoded images:", len(mask_lookup))
print("Classes:", classes)

assert len(mask_lookup) == 1200

Decoded images: 1200
Classes: ['defect_a', 'defect_b', 'defect_c', 'defect_d']


In [15]:
image_records = []

for image_id in image_ids:
    image_id = str(image_id)
    masks = mask_lookup[image_id]

    positive_classes = [
        c for c in classes
        if masks[c].sum() > 0
    ]

    combination = (
        "+".join(positive_classes)
        if positive_classes else "clean"
    )

    image_records.append({
        "image_id": image_id,
        "combination": combination,
        "is_clean": len(positive_classes) == 0,
        "positive_classes": positive_classes
    })

image_info = pd.DataFrame(image_records)

# Merge groups having fewer than 3 images
combination_counts = image_info["combination"].value_counts()

image_info["stratum"] = image_info["combination"].apply(
    lambda x: x if combination_counts[x] >= 3 else "rare_other"
)

# Stratified 80/20 split
train_ids, val_ids = train_test_split(
    image_info["image_id"],
    test_size=0.20,
    random_state=42,
    stratify=image_info["stratum"]
)

train_ids = set(train_ids)
val_ids = set(val_ids)

print("Total images:", len(image_info))
print("Train images:", len(train_ids))
print("Validation images:", len(val_ids))
print("Original combinations:", image_info["combination"].nunique())
print("Strata after rare merging:", image_info["stratum"].nunique())

assert len(train_ids) == 960
assert len(val_ids) == 240

Total images: 1200
Train images: 960
Validation images: 240
Original combinations: 15
Strata after rare merging: 12


In [16]:
# Q1(a): Completely clean images
clean_count = int(image_info["is_clean"].sum())

# Q1(b): Positive images for each defect class
class_positive_counts = {}

for c in classes:
    class_positive_counts[c] = sum(
        mask_lookup[image_id][c].sum() > 0
        for image_id in image_info["image_id"]
    )

# Q1(c): Unique original combination strings (before merging)
combination_counts = image_info["combination"].value_counts()
unique_combinations = image_info["combination"].nunique()

# Q1(d): Largest and smallest original combination groups
largest_group = int(combination_counts.max())
smallest_group = int(combination_counts.min())

print("Q1(a) Completely clean images:", clean_count)
print("Q1(b) Positive images per class:", class_positive_counts)
print("Q1(c) Unique combination strings:", unique_combinations)
print("Q1(d) Largest group:", largest_group)
print("Q1(d) Smallest group:", smallest_group)

Q1(a) Completely clean images: 578
Q1(b) Positive images per class: {'defect_a': np.int64(107), 'defect_b': np.int64(35), 'defect_c': np.int64(438), 'defect_d': np.int64(181)}
Q1(c) Unique combination strings: 15
Q1(d) Largest group: 578
Q1(d) Smallest group: 1


In [17]:
import numpy as np

defect_b_areas = []

for image_id in image_ids:
    mask = mask_lookup[image_id]["defect_b"]
    area = int(np.asarray(mask).sum())

    if area > 0:
        defect_b_areas.append(area)

total_pixels = 1200 * 102400
positive_pixels = sum(defect_b_areas)

percentage = (positive_pixels / total_pixels) * 100
mean_area = np.mean(defect_b_areas)
pos_weight = (total_pixels - positive_pixels) / positive_pixels

print(f"Q2(a) Positive pixel percentage: {percentage:.4f}%")
print(f"Q2(b) Mean defect_b area: {mean_area:.1f}")
print(f"Q2(c) Negative-to-positive ratio: {round(pos_weight)}")

Q2(a) Positive pixel percentage: 0.0160%
Q2(b) Mean defect_b area: 561.5
Q2(c) Negative-to-positive ratio: 6251


In [19]:
import numpy as np
import pandas as pd
from collections import Counter

# Native image dimensions
H, W = 128, 800
TOTAL_PIXELS = H * W
N_DRAWS = 1200

# Decode RLE: 1-based indexing, Fortran order
def decode_rle(rle, height=128, width=800):
    mask = np.zeros(height * width, dtype=np.uint8)

    if pd.isna(rle) or str(rle).strip() == "":
        return mask

    values = np.asarray(str(rle).split(), dtype=np.int64)
    starts = values[0::2] - 1
    lengths = values[1::2]

    for start, length in zip(starts, lengths):
        end = min(start + length, height * width)
        if start < height * width and end > start:
            mask[start:end] = 1

    return mask


# Build one defect-presence combination per image
classes = ["defect_a", "defect_b", "defect_c", "defect_d"]

image_groups = {}

for image_id, group in subset.groupby("image_id", sort=True):
    present = []

    for defect in classes:
        row = group[group["defect_type"] == defect]

        if len(row) > 0 and pd.notna(row.iloc[0]["mask_rle"]):
            mask = decode_rle(row.iloc[0]["mask_rle"])
            if mask.sum() > 0:
                present.append(defect)

    combination = "+".join(present) if present else "clean"
    image_groups[image_id] = combination


# Count original combination groups (before merging)
group_counts = Counter(image_groups.values())

n_clean = group_counts["clean"]

# Bounded square-root weight for each group
weights = {
    group: min(np.sqrt(n_clean / count), 5)
    for group, count in group_counts.items()
}

# Total weighted probability mass
total_mass = sum(
    group_counts[group] * weights[group]
    for group in group_counts
)

# Q3(a): Expected clean draw percentage
clean_mass = group_counts["clean"] * weights["clean"]
sampler_clean_pct = 100 * clean_mass / total_mass
natural_clean_pct = 100 * n_clean / 1200

# Q3(b): Expected defect_b-positive draws
b_groups = [
    group for group in group_counts
    if "defect_b" in group.split("+")
]

b_mass = sum(
    group_counts[group] * weights[group]
    for group in b_groups
)

sampler_b = N_DRAWS * b_mass / total_mass

b_positive_images = sum(group_counts[g] for g in b_groups)
natural_b = N_DRAWS * b_positive_images / 1200

# Q3(c): Enrichment factor
enrichment = sampler_b / natural_b

# Q3(d): Number of groups hitting weight cap of 5
cap_groups = sum(
    1 for group, count in group_counts.items()
    if weights[group] == 5
)

print("Combination groups:", len(group_counts))
print("Clean images:", n_clean)
print("Defect_b-positive images:", b_positive_images)

print("\nQ3(a)")
print(f"Sampler clean percentage: {sampler_clean_pct:.2f}%")
print(f"Natural clean percentage: {natural_clean_pct:.2f}%")

print("\nQ3(b)")
print(f"Sampler defect_b draws: {sampler_b:.2f}")
print(f"Natural defect_b draws: {natural_b:.2f}")

print("\nQ3(c)")
print(f"Enrichment factor: {enrichment:.2f}x")

print("\nQ3(d)")
print("Groups hitting cap of 5:", cap_groups)

Combination groups: 15
Clean images: 578
Defect_b-positive images: 35

Q3(a)
Sampler clean percentage: 28.81%
Natural clean percentage: 48.17%

Q3(b)
Sampler defect_b draws: 104.69
Natural defect_b draws: 35.00

Q3(c)
Enrichment factor: 2.99x

Q3(d)
Groups hitting cap of 5: 10


In [21]:

# ============================================================
# MILESTONE 4 - QUESTION 8
# 6-Epoch Baseline Training + Warmup/Cosine + Validation
# ============================================================

import os, glob, random, math
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# -------------------------
# 1. Reproducibility
# -------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# -------------------------
# 2. Dataset
# -------------------------
BASE = "/kaggle/input/competitions/industrial-surface-defect-detection/public"
CSV_PATH = os.path.join(BASE, "train.csv")
IMAGE_DIR = os.path.join(BASE, "train")

df = pd.read_csv(CSV_PATH)
df["image_id"] = df["image_id"].astype(str)

CLASSES = ["defect_a", "defect_b", "defect_c", "defect_d"]
H, W = 128, 800

# First 1200 unique image IDs in ASCII/alphabetical order
image_ids = sorted(df["image_id"].unique())[:1200]
subset = df[df["image_id"].isin(image_ids)].copy()
subset = subset.sort_values(["image_id", "defect_type"]).reset_index(drop=True)

assert len(image_ids) == 1200
assert len(subset) == 4800

# -------------------------
# 3. RLE decoding
# -------------------------
def decode_rle(rle, height=H, width=W):
    mask = np.zeros(height * width, dtype=np.uint8)

    if pd.isna(rle) or str(rle).strip() == "":
        return mask.reshape((height, width), order="F")

    values = np.asarray(str(rle).split(), dtype=np.int64)
    starts = values[0::2] - 1
    lengths = values[1::2]

    for start, length in zip(starts, lengths):
        end = min(start + length, height * width)
        if start < height * width and end > start:
            mask[start:end] = 1

    return mask.reshape((height, width), order="F")


# -------------------------
# 4. Build image-level masks
# -------------------------
mask_lookup = {}

for image_id, group in tqdm(subset.groupby("image_id"), desc="Decoding masks"):
    masks = []

    for cls in CLASSES:
        row = group[group["defect_type"] == cls]
        rle = row.iloc[0]["mask_rle"] if len(row) else np.nan
        masks.append(decode_rle(rle))

    mask_lookup[image_id] = np.stack(masks, axis=0)

# Combination strings
combination = {}

for image_id in image_ids:
    present = [
        CLASSES[i]
        for i in range(4)
        if mask_lookup[image_id][i].sum() > 0
    ]
    combination[image_id] = "+".join(present) if present else "clean"

strata = pd.Series(combination)
counts = strata.value_counts()
strata = strata.apply(lambda x: x if counts[x] >= 3 else "rare_other")

train_ids, val_ids = train_test_split(
    image_ids,
    test_size=0.20,
    random_state=42,
    stratify=strata.loc[image_ids].values
)

print("Train:", len(train_ids), "Validation:", len(val_ids))

# -------------------------
# 5. Locate image files
# -------------------------
image_files = {}

for path in glob.glob(os.path.join(IMAGE_DIR, "**", "*"), recursive=True):
    if os.path.isfile(path):
        stem = os.path.splitext(os.path.basename(path))[0]
        if stem not in image_files:
            image_files[stem] = path

missing = [x for x in image_ids if x not in image_files]
if missing:
    raise FileNotFoundError(
        f"{len(missing)} images not found. Example: {missing[:5]}"
    )

# -------------------------
# 6. Dataset class
# -------------------------
class SurfaceDataset(Dataset):
    def __init__(self, ids):
        self.ids = list(ids)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        image_id = self.ids[idx]

        image = Image.open(image_files[image_id]).convert("L")
        image = np.asarray(image, dtype=np.float32) / 255.0

        # Three-channel input for the baseline U-Net
        image = np.repeat(image[None, :, :], 3, axis=0)

        mask = mask_lookup[image_id].astype(np.float32)

        return torch.from_numpy(image), torch.from_numpy(mask)


train_loader = DataLoader(
    SurfaceDataset(train_ids),
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

val_loader = DataLoader(
    SurfaceDataset(val_ids),
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

# -------------------------
# 7. U-Net model
# -------------------------
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=4, base=16):
        super().__init__()

        self.enc1 = ConvBlock(in_channels, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.enc3 = ConvBlock(base * 2, base * 4)
        self.enc4 = ConvBlock(base * 4, base * 8)

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = ConvBlock(base * 8, base * 16)

        self.up4 = nn.ConvTranspose2d(base * 16, base * 8, 2, 2)
        self.dec4 = ConvBlock(base * 16, base * 8)

        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.dec3 = ConvBlock(base * 8, base * 4)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.dec2 = ConvBlock(base * 4, base * 2)

        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.dec1 = ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.out(d1)


model = UNet().to(device)
print("Model created successfully.")

# -------------------------
# 8. Loss and optimizer
# -------------------------
def dice_bce_loss(logits, targets):
    bce = F.binary_cross_entropy_with_logits(logits, targets)

    probs = torch.sigmoid(logits)
    dims = (0, 2, 3)

    intersection = (probs * targets).sum(dims)
    denominator = probs.sum(dims) + targets.sum(dims)

    dice = (2 * intersection + 1e-6) / (denominator + 1e-6)
    dice_loss = 1 - dice.mean()

    return 0.5 * bce + 0.5 * dice_loss


optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

EPOCHS = 6
STEPS_PER_EPOCH = len(train_loader)
TOTAL_STEPS = EPOCHS * STEPS_PER_EPOCH
WARMUP_STEPS = 60

print("Steps per epoch:", STEPS_PER_EPOCH)
print("Total optimizer steps:", TOTAL_STEPS)

# Warmup + cosine multiplier
def lr_multiplier(step):
    if step < WARMUP_STEPS:
        return max(1e-8, (step + 1) / WARMUP_STEPS)

    progress = (step - WARMUP_STEPS) / max(1, TOTAL_STEPS - WARMUP_STEPS)
    progress = min(progress, 1.0)

    return 0.5 * (1 + math.cos(math.pi * progress))


scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lr_lambda=lr_multiplier
)

amp_enabled = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

# -------------------------
# 9. Official validation metric
# -------------------------
@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    positive_dices = []
    empty_fp_scores = []
    empty_accuracies = []

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=amp_enabled
        ):
            logits = model(images)

        predictions = (torch.sigmoid(logits) >= 0.5).float()

        for i in range(len(images)):
            for c in range(4):
                gt = masks[i, c]
                pred = predictions[i, c]

                gt_sum = gt.sum().item()
                pred_sum = pred.sum().item()

                if gt_sum > 0:
                    dice = (
                        2 * (gt * pred).sum().item() + 1e-7
                    ) / (gt_sum + pred_sum + 1e-7)
                    positive_dices.append(dice)

                else:
                    fp_rate = pred_sum / (H * W)
                    empty_fp_scores.append(1 - fp_rate)
                    empty_accuracies.append(float(pred_sum == 0))

    foreground_dice = (
        np.mean(positive_dices) if positive_dices else 0.0
    )
    false_positive_score = (
        np.mean(empty_fp_scores) if empty_fp_scores else 0.0
    )
    empty_accuracy = (
        np.mean(empty_accuracies) if empty_accuracies else 0.0
    )

    official_score = (
        0.70 * foreground_dice
        + 0.20 * false_positive_score
        + 0.10 * empty_accuracy
    )

    return official_score


# -------------------------
# 10. Six-epoch training
# -------------------------
history = []
best_score = -1
best_epoch = None

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for images, masks in progress_bar:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=amp_enabled
        ):
            logits = model(images)
            loss = dice_bce_loss(logits, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.item()
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss = running_loss / len(train_loader)

    # Validation loss
    model.eval()
    val_running_loss = 0.0

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=amp_enabled
            ):
                logits = model(images)
                loss = dice_bce_loss(logits, masks)

            val_running_loss += loss.item()

    val_loss = val_running_loss / len(val_loader)
    val_score = evaluate(model, val_loader)
    current_lr = optimizer.param_groups[0]["lr"]

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "official_score": val_score,
        "lr": current_lr
    })

    if val_score > best_score:
        best_score = val_score
        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            "/kaggle/working/milestone4_best_model.pth"
        )

    print(
        f"Epoch {epoch+1}: "
        f"train_loss={train_loss:.4f}, "
        f"val_loss={val_loss:.4f}, "
        f"official_score={val_score:.4f}, "
        f"LR={current_lr:.7g}"
    )

# -------------------------
# 11. Results
# -------------------------
history_df = pd.DataFrame(history)

display(history_df)

print("Lowest validation loss epoch:",
      int(history_df["val_loss"].idxmin() + 1))

print("Best official score epoch:", best_epoch)
print("Best official score:", round(best_score, 3))

history_df.to_csv(
    "/kaggle/working/milestone4_q8_training_history.csv",
    index=False
)

print("Saved model: /kaggle/working/milestone4_best_model.pth")
print("Saved history: /kaggle/working/milestone4_q8_training_history.csv")


Device: cuda


Decoding masks:   0%|          | 0/1200 [00:00<?, ?it/s]

Train: 960 Validation: 240
Model created successfully.
Steps per epoch: 60
Total optimizer steps: 360


Epoch 1/6:   0%|          | 0/60 [00:00<?, ?it/s]

Epoch 1: train_loss=0.8525, val_loss=0.8432, official_score=0.1666, LR=0.0003


Epoch 2/6:   0%|          | 0/60 [00:00<?, ?it/s]

... (9 outputs left out)

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ba3353fb6a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionErrorException ignored in: : <function _MultiProcessingDataLoaderIter.__del__ at 0x7ba3353fb6a0>
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
can only test a child processTraceback (most recent call last):

    Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ba3353fb6a0>self._shutdown_workers()

Traceback (most recent call last):
  File "/usr/local/lib/pyt

Epoch 6: train_loss=0.7507, val_loss=0.7496, official_score=0.2500, LR=0


,epoch,train_loss,val_loss,official_score,lr
0,1,0.852509,0.843234,0.166613,0.000300
1,2,0.805390,0.796908,0.245359,0.000271
2,3,0.778405,0.771254,0.235321,0.000196
3,4,0.762075,0.759901,0.266638,0.000104
4,5,0.753695,0.749945,0.252012,0.000029
5,6,0.750715,0.749608,0.250038,0.000000


Lowest validation loss epoch: 6
Best official score epoch: 4
Best official score: 0.267
Saved model: /kaggle/working/milestone4_best_model.pth
Saved history: /kaggle/working/milestone4_q8_training_history.csv


In [22]:

import numpy as np
import pandas as pd

# ------------------------------------------------
# Q9: Calculate total defect area for every
# positive image-class pair in the 1200-image subset
# ------------------------------------------------

small = 0
medium = 0
large = 0

areas = []

for image_id in image_ids:
    masks = mask_lookup[image_id]

    for class_idx, class_name in enumerate(CLASSES):
        area = int(masks[class_idx].sum())

        # Only positive image-class pairs
        if area > 0:
            areas.append({
                "image_id": image_id,
                "class": class_name,
                "area": area
            })

            if 1 <= area <= 100:
                small += 1
            elif 101 <= area <= 500:
                medium += 1
            else:
                large += 1

# ------------------------------------------------
# Part A: Counts per bin
# ------------------------------------------------

areas_df = pd.DataFrame(areas)

total_positive_pairs = len(areas_df)
total_positive_pixels = areas_df["area"].sum()

small_pixels = areas_df.loc[
    areas_df["area"].between(1, 100), "area"
].sum()

medium_pixels = areas_df.loc[
    areas_df["area"].between(101, 500), "area"
].sum()

large_pixels = areas_df.loc[
    areas_df["area"] >= 501, "area"
].sum()

# ------------------------------------------------
# Part B: Percentage of pixels in 501+ bin
# ------------------------------------------------

large_pixel_percentage = (
    large_pixels / total_positive_pixels * 100
)

# ------------------------------------------------
# Part C: Expected missed positive pairs
# ------------------------------------------------

missed_small = small * (1 - 0.30)
missed_medium = medium * (1 - 0.70)
missed_large = large * (1 - 0.95)

expected_missed = (
    missed_small + missed_medium + missed_large
)

missed_by_bin = {
    "1-100 px": missed_small,
    "101-500 px": missed_medium,
    "501+ px": missed_large
}

largest_missed_bin = max(
    missed_by_bin, key=missed_by_bin.get
)

# ------------------------------------------------
# Print answers
# ------------------------------------------------

print("Q9(a) Positive pairs:")
print("1-100 px:", small)
print("101-500 px:", medium)
print("501+ px:", large)

print("\nQ9(b) Percentage of positive pixels in 501+ bin:")
print(f"{large_pixel_percentage:.2f}%")

print("\nQ9(c) Expected missed positive pairs:")
print(f"{expected_missed:.2f}")

print("Missed pairs by bin:")
for name, value in missed_by_bin.items():
    print(f"{name}: {value:.2f}")

print("Bin contributing most missed pairs:", largest_missed_bin)

# Optional detailed table
display(areas_df.head(10))


Q9(a) Positive pairs:
1-100 px: 5
101-500 px: 99
501+ px: 657

Q9(b) Percentage of positive pixels in 501+ bin:
99.24%

Q9(c) Expected missed positive pairs:
66.05
Missed pairs by bin:
1-100 px: 3.50
101-500 px: 29.70
501+ px: 32.85
Bin contributing most missed pairs: 501+ px


,image_id,class,area
0,000HAM628W,defect_c,584
1,005VFS2MW4,defect_d,11172
2,0070O9IJVX,defect_d,3748
3,00761I10TU,defect_c,4635
4,010Y3674EF,defect_c,866
5,011ZK8OPYS,defect_c,9983
6,011ZK8OPYS,defect_d,7232
7,01AOH74AT9,defect_d,5421
8,01DQK3XK45,defect_c,890
9,01K0RZAGSL,defect_c,4085


In [23]:
import os
import gc
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import wandb

from torch.utils.data import DataLoader, WeightedRandomSampler

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 16
EPOCHS = 6
BASE_LR = 3e-4
NUM_CLASSES = 4
IMAGE_SIZE = (128, 800)

WANDB_PROJECT = "24f2006741-t32026"

EXPERIMENTS = {
    "R0":  {"seed": 42, "scheduler": "constant", "sampler": "natural"},
    "R0b": {"seed": 7,  "scheduler": "constant", "sampler": "natural"},
    "R1":  {"seed": 42, "scheduler": "cosine",   "sampler": "natural"},
    "R2":  {"seed": 42, "scheduler": "constant", "sampler": "sqrt"},
}

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

print("Device:", DEVICE)
print("Experiments:", list(EXPERIMENTS))

Device: cuda
Experiments: ['R0', 'R0b', 'R1', 'R2']


In [24]:
from collections import Counter

def make_sqrt_sampler(train_ids, combination_map):
    group_counts = Counter(
        combination_map[image_id] for image_id in train_ids
    )

    clean_count = group_counts["clean"]

    group_weights = {
        group: min(np.sqrt(clean_count / count), 5)
        for group, count in group_counts.items()
    }

    sample_weights = [
        group_weights[combination_map[image_id]]
        for image_id in train_ids
    ]

    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(train_ids),
        replacement=True
    )

    print("Combination groups:", len(group_counts))
    print("Groups hitting cap:", sum(
        weight >= 5 for weight in group_weights.values()
    ))

    return sampler

In [25]:
import math

def make_scheduler(optimizer, scheduler_type, steps_per_epoch):
    total_steps = EPOCHS * steps_per_epoch
    warmup_steps = 60

    if scheduler_type == "constant":
        return None

    def lr_multiplier(step):
        if step < warmup_steps:
            return step / warmup_steps

        progress = (step - warmup_steps) / (
            total_steps - warmup_steps
        )

        return 0.5 * (1 + math.cos(math.pi * progress))

    return torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lr_multiplier
    )

In [26]:
# ADAPTERS:
# build_model()          -> returns a fresh baseline model
# make_train_loader()    -> returns natural-sampling train loader
# make_val_loader()      -> returns the same validation loader
# train_one_epoch(...)   -> trains one epoch and returns train loss
# validate_one_epoch(...) -> returns (validation loss, official score)

# Do not run until these five functions are connected to your Q8 code.

def run_experiment(name, config):

    print(f"\n{'='*55}")
    print("Starting:", name, config)
    print(f"{'='*55}")

    set_seed(config["seed"])

    model = build_model().to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=BASE_LR
    )

    # Use the natural loader for R0, R0b and R1.
    # R2 uses the bounded square-root sampler.
    if config["sampler"] == "sqrt":
        sampler = make_sqrt_sampler(train_ids, combination_map)
        train_loader = make_train_loader(sampler=sampler)
    else:
        train_loader = make_train_loader(sampler=None)

    val_loader = make_val_loader()

    steps_per_epoch = len(train_loader)

    scheduler = make_scheduler(
        optimizer,
        config["scheduler"],
        steps_per_epoch
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(DEVICE.type == "cuda")
    )

    history = []

    run = wandb.init(
        project=WANDB_PROJECT,
        name=f"Milestone4-{name}",
        config={
            **config,
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "learning_rate": BASE_LR,
            "amp": True
        },
        reinit="finish"
    )

    best_score = -float("inf")
    best_model_state = None

    for epoch in range(EPOCHS):

        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            scaler=scaler,
            device=DEVICE,
            scheduler=scheduler
        )

        val_loss, official_score = validate_one_epoch(
            model=model,
            loader=val_loader,
            device=DEVICE
        )

        if official_score > best_score:
            best_score = official_score
            best_model_state = copy.deepcopy(model.state_dict())

        current_lr = optimizer.param_groups[0]["lr"]

        row = {
            "run": name,
            "epoch": epoch + 1,
            "train_loss": float(train_loss),
            "val_loss": float(val_loss),
            "official_score": float(official_score),
            "lr": float(current_lr)
        }

        history.append(row)
        wandb.log(row)

        print(
            f"{name} | Epoch {epoch+1}/{EPOCHS} | "
            f"Train: {train_loss:.4f} | "
            f"Val: {val_loss:.4f} | "
            f"Score: {official_score:.4f} | "
            f"LR: {current_lr:.7f}"
        )

    pd.DataFrame(history).to_csv(
        f"/kaggle/working/milestone4_{name}_history.csv",
        index=False
    )

    torch.save(
        best_model_state,
        f"/kaggle/working/milestone4_{name}_best.pth"
    )

    wandb.summary["best_official_score"] = best_score
    run.finish()

    del model, optimizer, train_loader, val_loader
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return history

In [29]:
def build_model():
    return UNet()

In [31]:
# ============================================================
# MILESTONE 4 - QUESTION 10: CONTROLLED EXPERIMENTS (OFAT)
# R0, R0b, R1, R2
# ============================================================

import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler
import math

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EPOCHS = 6
BATCH_SIZE = 16
BASE_LR = 3e-4
WARMUP_STEPS = 60

# ------------------------------------------------------------
# 1. Model and loaders
# ------------------------------------------------------------

def build_model():
    return UNet().to(DEVICE)


def make_train_loader(sampler=None):
    dataset = SurfaceDataset(train_ids)

    if sampler is not None:
        return DataLoader(
            dataset,
            batch_size=BATCH_SIZE,
            sampler=sampler,
            num_workers=0,
            pin_memory=(DEVICE.type == "cuda")
        )

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda")
    )


def make_val_loader():
    return DataLoader(
        SurfaceDataset(val_ids),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda")
    )


# ------------------------------------------------------------
# 2. Bounded square-root sampler for R2
# ------------------------------------------------------------

def make_sqrt_sampler(train_ids, combination_map):
    train_groups = [combination_map[x] for x in train_ids]
    group_counts = pd.Series(train_groups).value_counts()

    n_clean = int(group_counts.get("clean", 0))

    if n_clean == 0:
        raise ValueError("No clean images found in training split.")

    group_weights = {
        group: min(math.sqrt(n_clean / count), 5)
        for group, count in group_counts.items()
    }

    sample_weights = [
        group_weights[combination_map[x]]
        for x in train_ids
    ]

    return WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(train_ids),
        replacement=True
    )


# ------------------------------------------------------------
# 3. Training and validation
# ------------------------------------------------------------

def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    running_loss = 0.0

    amp_enabled = DEVICE.type == "cuda"

    for images, masks in loader:
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=amp_enabled
        ):
            logits = model(images)
            loss = dice_bce_loss(logits, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

    return running_loss / len(loader)


@torch.no_grad()
def validate_one_epoch(model, loader):
    model.eval()
    running_loss = 0.0
    empty_pairs = 0
    false_alarm_pairs = 0

    amp_enabled = DEVICE.type == "cuda"

    for images, masks in loader:
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=amp_enabled
        ):
            logits = model(images)
            loss = dice_bce_loss(logits, masks)

        running_loss += loss.item()

        predictions = (torch.sigmoid(logits) >= 0.5).float()

        for i in range(len(images)):
            for c in range(4):
                if masks[i, c].sum().item() == 0:
                    empty_pairs += 1

                    if predictions[i, c].sum().item() > 0:
                        false_alarm_pairs += 1

    val_loss = running_loss / len(loader)
    official_score = evaluate(model, loader)

    false_alarm_rate = (
        100 * false_alarm_pairs / empty_pairs
        if empty_pairs else 0.0
    )

    return val_loss, official_score, false_alarm_rate


# ------------------------------------------------------------
# 4. Run one experiment
# ------------------------------------------------------------

def run_experiment(name, config):

    print("\n" + "=" * 55)
    print("Starting:", name, config)
    print("=" * 55)

    seed = config["seed"]
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = build_model()

    if config["sampler"] == "sqrt":
        sampler = make_sqrt_sampler(train_ids, combination)
        train_loader = make_train_loader(sampler=sampler)
    else:
        train_loader = make_train_loader()

    val_loader = make_val_loader()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=BASE_LR
    )

    steps_per_epoch = len(train_loader)
    total_steps = EPOCHS * steps_per_epoch

    scheduler = None

    if config["scheduler"] == "warmcos":
        def lr_multiplier(step):
            if step < WARMUP_STEPS:
                return max(1e-8, (step + 1) / WARMUP_STEPS)

            progress = (
                (step - WARMUP_STEPS)
                / max(1, total_steps - WARMUP_STEPS)
            )
            progress = min(progress, 1.0)

            return 0.5 * (1 + math.cos(math.pi * progress))

        scheduler = torch.optim.lr_scheduler.LambdaLR(
            optimizer,
            lr_lambda=lr_multiplier
        )

    amp_enabled = DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=amp_enabled
    )

    history = []

    for epoch in range(EPOCHS):

        train_loss = train_one_epoch(
            model, train_loader, optimizer, scaler
        )

        val_loss, score, false_alarm = validate_one_epoch(
            model, val_loader
        )

        if scheduler is not None:
            scheduler.step()

        lr = optimizer.param_groups[0]["lr"]

        row = {
            "run": name,
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "official_score": score,
            "false_alarm_rate": false_alarm,
            "lr": lr
        }

        history.append(row)

        print(
            f"{name} | Epoch {epoch+1}/{EPOCHS} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"score={score:.4f} | "
            f"empty_false_alarm={false_alarm:.2f}% | "
            f"LR={lr:.7g}"
        )

    return history


# ------------------------------------------------------------
# 5. Four controlled experiments
# ------------------------------------------------------------

EXPERIMENTS = {
    "R0": {
        "seed": 42,
        "scheduler": "constant",
        "sampler": "natural"
    },
    "R0b": {
        "seed": 7,
        "scheduler": "constant",
        "sampler": "natural"
    },
    "R1": {
        "seed": 42,
        "scheduler": "warmcos",
        "sampler": "natural"
    },
    "R2": {
        "seed": 42,
        "scheduler": "constant",
        "sampler": "sqrt"
    }
}

all_histories = []

for name, config in EXPERIMENTS.items():
    history = run_experiment(name, config)
    all_histories.extend(history)

results_df = pd.DataFrame(all_histories)

results_df.to_csv(
    "/kaggle/working/milestone4_q10_all_results.csv",
    index=False
)

display(results_df)


# ------------------------------------------------------------
# 6. Best score and false-alarm summary per run
# ------------------------------------------------------------

summary_df = (
    results_df.groupby("run")
    .agg(
        best_official_score=("official_score", "max"),
        empty_false_alarm_rate=("false_alarm_rate", "last")
    )
    .reset_index()
)

display(summary_df)

print("Saved: /kaggle/working/milestone4_q10_all_results.csv")


Starting: R0 {'seed': 42, 'scheduler': 'constant', 'sampler': 'natural'}
R0 | Epoch 1/6 | train_loss=0.8319 | val_loss=0.8310 | score=0.1676 | empty_false_alarm=100.00% | LR=0.0003
R0 | Epoch 2/6 | train_loss=0.7903 | val_loss=0.7754 | score=0.2416 | empty_false_alarm=78.96% | LR=0.0003
R0 | Epoch 3/6 | train_loss=0.7623 | val_loss=0.7530 | score=0.2672 | empty_false_alarm=75.25% | LR=0.0003
R0 | Epoch 4/6 | train_loss=0.7368 | val_loss=0.7231 | score=0.2675 | empty_false_alarm=65.47% | LR=0.0003
R0 | Epoch 5/6 | train_loss=0.7104 | val_loss=0.6976 | score=0.2894 | empty_false_alarm=32.18% | LR=0.0003
R0 | Epoch 6/6 | train_loss=0.6852 | val_loss=0.6729 | score=0.3146 | empty_false_alarm=6.93% | LR=0.0003

Starting: R0b {'seed': 7, 'scheduler': 'constant', 'sampler': 'natural'}
R0b | Epoch 1/6 | train_loss=0.8357 | val_loss=0.8107 | score=0.1806 | empty_false_alarm=91.09% | LR=0.0003
R0b | Epoch 2/6 | train_loss=0.7982 | val_loss=0.7878 | score=0.2416 | empty_false_alarm=69.18% | LR=0

,run,epoch,train_loss,val_loss,official_score,false_alarm_rate,lr
0,R0,1,0.831905,0.830994,0.167616,100.000000,0.000300
1,R0,2,0.790321,0.775379,0.241641,78.960396,0.000300
2,R0,3,0.762316,0.752998,0.267225,75.247525,0.000300
3,R0,4,0.736810,0.723123,0.267504,65.470297,0.000300
4,R0,5,0.710410,0.697626,0.289359,32.178218,0.000300
5,R0,6,0.685185,0.672922,0.314650,6.930693,0.000300
6,R0b,1,0.835670,0.810736,0.180611,91.089109,0.000300
7,R0b,2,0.798187,0.787832,0.241608,69.183168,0.000300
8,R0b,3,0.772470,0.760118,0.274575,33.787129,0.000300
9,R0b,4,0.748127,0.738105,0.300040,9.777228,0.000300


,run,best_official_score,empty_false_alarm_rate
0,R0,0.314650,6.930693
1,R0b,0.308038,3.589109
2,R1,0.184846,100.000000
3,R2,0.278509,40.594059


Saved: /kaggle/working/milestone4_q10_all_results.csv
